[Guide Index](README.md)


# 03 SPARQL

`StarLayerGraph` runs SPARQL 1.2 on Oxigraph, including the use of Turtle 1.2-based shorthand syntax. 


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph import StarLayerGraph

EX = "http://example.org/"
RDF = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"

## 1. Running a query

`query()` runs a SPARQL query on the graph and returns a `Result`; what it holds depends on the query form. 

- **SELETCT** returns a series of rows.  A row gives a variable as an attribute (`row.p`) or a key (`row["p"]`). `result.bindings` gives the same rows as a list of dicts, one per row, each mapping a variable name to its term (a variable left unbound in a row is missing from its dict).

- **ASK** returns a boolean (true/false).  

- **CONSTRUCT** and **DESCRIBE** (section 2 below) returns a set of triples and a `StarLayerGraph`.



In [2]:
g_team = StarLayerGraph()
# the graph keeps the prefixes the data declares, so queries can use ex: without a PREFIX line
g_team.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:teamA ex:name "ann", "al" .
    ex:teamB ex:name "bo" .
""", format="turtle")

# SELECT: query() returns a Result; iterate it for the rows
result = g_team.query("SELECT ?team ?n WHERE { ?team ex:name ?n } ORDER BY ?n")
print("Select Rows:")
for row in result:
    print(row.team.value, row["n"].value)

print("\nBindings:")
print(result.bindings)

# ASK: the Result is true or false
result = g_team.query("ASK { ?s ex:name 'bo' }")
print("\nAsk Response:")
print(bool(result))

# CONSTRUCT: .graph is a new StarLayerGraph of the triples the template built
result = g_team.query("CONSTRUCT { ?t ex:called ?n } WHERE { ?t ex:name ?n }")
built = result.graph
print("\nConstruct Triples: ")
print(built.serialize(format="turtle"))


Select Rows:
http://example.org/teamA al
http://example.org/teamA ann
http://example.org/teamB bo

Bindings:
[{'team': <NamedNode value=http://example.org/teamA>, 'n': <Literal value=al datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>>}, {'team': <NamedNode value=http://example.org/teamA>, 'n': <Literal value=ann datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>>}, {'team': <NamedNode value=http://example.org/teamB>, 'n': <Literal value=bo datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>>}]

Ask Response:
True

Construct Triples: 
<http://example.org/teamA> <http://example.org/called> "ann" , "al" .
<http://example.org/teamB> <http://example.org/called> "bo" .



## 2. Describing a resource

`DESCRIBE` returns the triples that describe a resource (or resources). 

CONSTRUCT builds triples from a template provided in the query. DESCRIBE, as defined in the SPARQL 1.2 specification, has no pre-defined template.  Oxigraph returns the triples that have the resource as their subject. 

`DESCRIBE` takes an IRI or variables bound by a `WHERE` clause. The result is read like a CONSTRUCT result: `result.graph` or `result.triples`.

In [3]:
# DESCRIBE an IRI (g_team is the graph from section 1)
result = g_team.query("DESCRIBE ex:teamA")
print("Describe an IRI:")
print(result.graph.serialize(format="turtle"))

# DESCRIBE a variable bound by WHERE
result = g_team.query("DESCRIBE ?team WHERE { ?team ex:name 'bo' }")
print("Describe a variable:")
print(result.graph.serialize(format="turtle"))


Describe an IRI:
<http://example.org/teamA> <http://example.org/name> "ann" , "al" .

Describe a variable:
<http://example.org/teamB> <http://example.org/name> "bo" .



## 3. Writing query results

`Result.serialize(format=None)` writes a result. 

A SELECT or ASK result is written as SPARQL Query Results `"json"` (the default), `"xml"`, `"csv"` or `"tsv"`; the format can be a name, a media type such as `"text/csv"`, or a file extension. A CONSTRUCT or DESCRIBE result is written as RDF, same as `StarLayerGraph.serialize` (Turtle by default). 

In [4]:
result = g_team.query("SELECT ?team ?n WHERE { ?team ex:name ?n } ORDER BY ?n")
print("csv format")
print(result.serialize(format="csv"))
print("tsv format")
print(result.serialize(format="tsv"))
print("json format")
print(g_team.query("ASK { ?s ex:name 'bo' }").serialize())

# CONSTRUCT is written as RDF; 
constructed = g_team.query("CONSTRUCT { ?t ex:called ?n } WHERE { ?t ex:name ?n }")
print ("\nn-triples format")
print("\n".join(sorted(constructed.serialize(format="nt").splitlines())))




csv format
team,n
http://example.org/teamA,al
http://example.org/teamA,ann
http://example.org/teamB,bo

tsv format
?team	?n
<http://example.org/teamA>	"al"
<http://example.org/teamA>	"ann"
<http://example.org/teamB>	"bo"

json format
{"head": {}, "boolean": true}

n-triples format
<http://example.org/teamA> <http://example.org/called> "al" .
<http://example.org/teamA> <http://example.org/called> "ann" .
<http://example.org/teamB> <http://example.org/called> "bo" .


## 4. The example graph

Sections 4 to 8 query one graph built here.  It has a name in two languages with a base direction, and the reifications and annotations that the later sections query. It uses the `ex` and `rdf` prefixes..


In [5]:
g_parsed = StarLayerGraph()
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals: AcmeCorp's name in English and Arabic
    ex:AcmeCorp ex:name "Acme Corporation"@en--ltr .
    ex:AcmeCorp ex:name "شركة أكمي"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> ;
      ex:source ex:HRSystem ;
      ex:confidence "high" .

    # anonymous inline annotation block (asserted triple)
    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .

    # named reifier with annotations
    ex:alice ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} .

    # named reifier without annotation block
    ex:alice ex:worksWith ex:SalesTeam ~ ex:stmt2 .
    ex:stmt2 ex:asReported ex:HRSystem .
    ex:stmt2 ex:reportedDate "1/1/2026" .

    # reification shorthand << s p o >> (no parens): an anonymous, unasserted
    # reifier of the same base triple ex:claim reifies above - a triple
    # can have more than one independent reification
    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:LinkedIn .

    # unasserted quoted triple term
    ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
''', format='turtle')
print("parsed triples:", len(g_parsed))
print()
# pretty=True is StarLayer's pretty Turtle writer (grouped subjects, {| |} annotation blocks);
# the default serialize() is Oxigraph's flat one-statement-per-line Turtle.
print(g_parsed.serialize(format="turtle", pretty=True))

parsed triples: 20

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:AcmeCorp ex:name "Acme Corporation"@en--ltr, "شركة أكمي"@ar--rtl .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} ;
    ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} ;
    ex:worksWith ex:SalesTeam ~ ex:stmt2 {| ex:asReported ex:HRSystem ; ex:reportedDate "1/1/2026" |} .

<< ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:confidence "high" ;
    ex:source ex:HRSystem .

<< ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:LinkedIn .



## 5. Triple functions

SPARQL 1.2 introduces now functions for working with triple terms.

- `isTRIPLE(x)` — true when a value is a triple term
- `TRIPLE(s, p, o)` — a functional way to write a triple term `<<( s p o )>>`
- `SUBJECT()`, `PREDICATE()`, `OBJECT()` — extract the parts out of a triple term

In [6]:
# 5.1 detect triple-term values using isTRIPLE
print("5.1 isTRIPLE()")
rows = g_parsed.query("""
SELECT ?claim ?p ?statement WHERE {
  ?claim ?p ?statement .
  FILTER( isTRIPLE(?statement))
  FILTER( ?p != rdf:reifies)
}
""")

# anonymous reifiers have a new label on every parse, so sort the printed lines
for line in sorted(
    f"{g_parsed.qname_term(row.claim)} {g_parsed.qname_term(row.p)} {g_parsed.qname_term(row.statement)}"
    for row in rows
):
    print(line)


5.1 isTRIPLE()
ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>>


In [7]:
# 5.2 TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
print("5.2 TRIPLE()/SUBJECT()/PREDICATE()/OBJECT()")
rows = g_parsed.query("""
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:alice, ex:worksFor, ex:AcmeCorp))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname_term(row.s), g_parsed.qname_term(row.p), g_parsed.qname_term(row.o))


5.2 TRIPLE()/SUBJECT()/PREDICATE()/OBJECT()
ex:alice ex:worksFor ex:AcmeCorp


## 6. Triple-term and annotation syntax in queries

The Turtle 1.2 syntax used to parse and serialize graphs also works in a SPARQL 1.2 query.

`<<( s p o )>>` can be used to represent a triple term.  The shorthand versions `{| ?pred ?val |}`, `~ ?r` and `<< s p o >>`, which stands for the `rdf:reifies` pattern can also be used.  

In [8]:
# 6.1 <<( ?s ?p ?o )>>: bind all terms inside a triple term
print("6.1 <<( ?s ?p ?o )>> pattern")
rows = g_parsed.query("""
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:worksFor)
}
ORDER BY ?claim ?s ?o
""")

# ?claim can be an anonymous reifier (a BlankNode) as well as a named one (a NamedNode);
# qname_term() writes either: a prefixed name for an IRI, _:label for a blank node.
for row in rows:
    print(
        g_parsed.qname_term(row.claim),
        g_parsed.qname_term(row.s),
        g_parsed.qname_term(row.p),
        g_parsed.qname_term(row.o),
        "Source:", g_parsed.qname_term(row.source),
    )

6.1 <<( ?s ?p ?o )>> pattern
_:d2eb9a0e233b7975d056f489b7498e4a ex:alice ex:worksFor ex:AcmeCorp Source: ex:LinkedIn
ex:claim ex:alice ex:worksFor ex:AcmeCorp Source: ex:HRSystem


In [9]:
# 6.2 <<( s p o )>> can also be written as a value in an expression such as BIND(),
# not only as the object of a triple pattern. It is equivalent to TRIPLE(s, p, o) in section 5.
print()
print("6.2 <<( s p o )>> in BIND()")
rows = g_parsed.query("""
SELECT ?tt WHERE {
  BIND( <<( ex:alice ex:worksFor ex:AcmeCorp )>> AS ?tt )
}
""")
for row in rows:
    # a pyoxigraph Triple: a triple term is a value of its own, with .subject/.predicate/.object
    print(type(row.tt).__name__, row.tt)


6.2 <<( s p o )>> in BIND()
Triple <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp>


In [10]:
# 6.3 {| ?pred ?val |}: query an anonymous reifier's annotations inline (asserted triple)
# The reifier also has an rdf:reifies triple, which the FILTER leaves out.
print("6.3 {| ?pred ?val |}")
rows = g_parsed.query("""
SELECT ?pred ?val WHERE {
  ex:alice ex:likes ex:ProductABC {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname_term(row.pred), g_parsed.qname_term(row.val))

# 6.4 ~ ?r: bind the reifier itself for a named reifier
print()
print("6.4 ~ ?r")
rows = g_parsed.query("""
SELECT ?r WHERE {
  ex:alice ex:worksWith ex:SalesTeam ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname_term(row.r))

# 6.5 << s p o >> ?pred ?val: reification shorthand, no assertion required
# No FILTER here, so the rdf:reifies triple of the reifier is in the answer.
print()
print("6.5 << s p o >>")
rows = g_parsed.query("""
SELECT ?pred ?val WHERE {
  << ex:alice ex:likes ex:ProductABC >> ?pred ?val
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname_term(row.pred), g_parsed.qname_term(row.val))

6.3 {| ?pred ?val |}
ex:since "2020"
ex:source ex:CRM

6.4 ~ ?r
ex:stmt2

6.5 << s p o >>
ex:since "2020"
ex:source ex:CRM
rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>>


## 7. Language and direction functions

- `LANG()`, `hasLANG()` — read and check a literal's language tag; `LANG()` is a SPARQL 1.1 function, now also aware of direction-tagged literals
- `LANGDIR()`, `hasLANGDIR()` — read and check a literal's base direction
- `STRLANGDIR()` — build a direction-tagged literal

In [11]:
# 7.1 LANG() / hasLANG() / LANGDIR() / hasLANGDIR() over AcmeCorp's two names
print("7.1 LANG()/hasLANG()/LANGDIR()/hasLANGDIR()")
rows = g_parsed.query("""
SELECT ?s ?lang ?hasLang ?dir ?hasDir WHERE {
  ?s ex:name ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(hasLANG(?lit) AS ?hasLang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s ?lang
""")
for row in rows:
    # row.lang and row.dir are Literals; .value is the Python str
    print(g_parsed.qname_term(row.s), row.lang.value, row.hasLang.value, row.dir.value, row.hasDir.value)

# 7.2 STRLANGDIR() constructs a direction-tagged literal directly from plain strings
print()
print("7.2 STRLANGDIR()")
rows = g_parsed.query("""
SELECT ?lit WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }
""")
for row in rows:
    print(row.lit, "| language:", row.lit.language, "| direction:", row.lit.direction)


7.1 LANG()/hasLANG()/LANGDIR()/hasLANGDIR()
ex:AcmeCorp ar true rtl true
ex:AcmeCorp en true ltr true

7.2 STRLANGDIR()
"hi"@en--ltr | language: en | direction: ltr


## 8. The `VERSION "1.2"` query prologue directive

Queries are often stored, shared, or sent to other stores, and not every store supports SPARQL 1.2. A leading `VERSION` line (before any `PREFIX` or `SELECT`) states which version of SPARQL a query needs. It is optional.

- A query that is limited to run on a SPARQL 1.1 store can declare `VERSION "1.1"`. If the query is later edited to add a SPARQL 1.2 feature to it, such as a triple term, StarLayer rejects the query with a `ValueError` when you run it, so the mistake is found in StarLayer and not when the store rejects it.
- A query that relies on SPARQL 1.2 can declare `VERSION "1.2"`, so the requirement can be included in the query itself.

A query with no `VERSION` line is not checked.

StarLayer rejects a query whose `VERSION` line is contrqadicted by its text or that names an unknown version, and it does so before anything is sent to a store. A graph backed by an HTTP endpoint (see the [backend guide](06b-backend-graph-databases.ipynb)) checks the `VERSION` line before anything is sent. What a remote store does with a `VERSION` line is controlld by that that store. 

In [12]:
# 8.1 VERSION "1.2" declared, and the query does use RDF 1.2 syntax - accepted.
# ?stmt can match an anonymous reifier (a BlankNode) as well as a named one,
# so print via qname_term(), which handles either.
print("8.1 VERSION \"1.2\" matches usage")
rows = g_parsed.query("""
VERSION "1.2"
SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")
print(sorted(g_parsed.qname_term(row.stmt) for row in rows))

# 8.2 VERSION "1.1" declared, but the query text itself uses a triple term - rejected.
# The check is StarLayer's and runs before the query is run or sent anywhere.
print()
print("8.2 VERSION \"1.1\" mismatch is rejected")
try:
    g_parsed.query("""
    VERSION "1.1"
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
    """)
except ValueError as error:
    print(type(error).__name__ + ":", error)


8.1 VERSION "1.2" matches usage
['_:d2eb9a0e233b7975d056f489b7498e4a', 'ex:claim']

8.2 VERSION "1.1" mismatch is rejected
SPARQL12ConformanceError: SPARQL query declares VERSION '1.1' but uses a triple term, which '1.1' conformance excludes (SPARQL 1.2 Query sec 4.3)


## 9. What an answer depends on

`graph.relevant_subgraph(query)` returns the triples of the graph that the answer to a query depends on, as a new graph. Every way the answer can be derived is kept. It takes a SELECT, ASK or CONSTRUCT query.

On a dataset it returns a dataset of quads, each in the graph it was read from. A `GRAPH` block keeps the quads of the graph it names. A pattern outside a `GRAPH` block reads the default graph (the union of all graphs for a `default_union` dataset), and keeps a quad in every graph that holds it.

In [13]:
from starlayer.graph import StarLayerDataset

evidence = StarLayerGraph()
evidence.parse(data='''@prefix ex: <http://example.org/> .
ex:teamA ex:name "ann" ; ex:size 2 .
ex:teamB ex:name "bo" ; ex:size 9 .''', format="turtle")
used = evidence.relevant_subgraph("SELECT ?t WHERE { ?t ex:name 'bo' ; ex:size ?n FILTER(?n > 5) }")
print(len(evidence), "triples in the graph;", len(used), "that the answer depends on:")
for s, p, o in sorted(used.triples(), key=str):
    print(" ", s.value, p.value, o.value)

# a dataset gives a dataset of quads
dataset = StarLayerDataset()
dataset.parse(data='''@prefix ex: <http://example.org/> .
ex:teamA ex:size 2 .
ex:teamB ex:size 9 .
ex:archive { ex:teamB ex:size 9 . ex:teamC ex:size 1 }''', format="trig")
used = dataset.relevant_subgraph("SELECT ?t WHERE { ?t ex:size ?n FILTER(?n > 5) }")
print()
print(len(list(dataset.quads())), "quads in the dataset;", len(list(used.quads())), "that the answer depends on:")
for q in sorted(used.quads(), key=str):
    print(" ", q.subject.value, q.predicate.value, q.object.value, "| graph:", q.graph_name)

# a GRAPH block keeps the quads of the graph it names
print("\nDataset exampe using a GRAPH block:")
used = dataset.relevant_subgraph("SELECT ?t WHERE { GRAPH ex:archive { ?t ex:size ?n FILTER(?n > 5) } }")

for q in sorted(used.quads(), key=str):
    print(" ", q.subject.value, q.predicate.value, q.object.value, "| graph:", q.graph_name)


4 triples in the graph; 2 that the answer depends on:
  http://example.org/teamB http://example.org/name bo
  http://example.org/teamB http://example.org/size 9

4 quads in the dataset; 2 that the answer depends on:
  http://example.org/teamB http://example.org/size 9 | graph: DEFAULT
  http://example.org/teamB http://example.org/size 9 | graph: <http://example.org/archive>

Dataset exampe using a GRAPH block:
  http://example.org/teamB http://example.org/size 9 | graph: <http://example.org/archive>


## 10. Preparing a query for re-use

`StarQuery(text, ...)` from `starlayer.sparql` allows for a query object to be created and then run against a graphs with `run(graph)`. It is for a query that an application creates to be run against many graphs. A syntax error is raised when the query is created.

The options of `query()` are bound once when the query is made. The ones you will use most often:

- `initNs`: the prefixes the query text uses, so the text needs no `PREFIX` lines.
- `initBindings`: variables that are bound before the query runs.
- `entailment`: the entailment regime to query under (see the [inferencing guide](02e-graphs-inferencing.ipynb)).
- `timeout`: the seconds the OWL-Direct reasoner may take, which applies to that regime only.

`base`, `infer` and `engine` are also accepted, as for `query()`. Any option can be changed for a single run, such as `run(graph, entailment=None)` to run without entailment. The query itself keeps its bound value.

In [14]:
from starlayer.graph import ENTAILMENT, NamedNode
from starlayer.sparql import StarQuery

# Two graphs to run the queries against.
# people: ann is a Member.
people = StarLayerGraph()
people.parse(data="@prefix ex: <http://example.org/> . ex:ann a ex:Member .", format="turtle")
# staff: bo is a Chair, cy and di are Clerks, and a Chair is a kind of Member.
staff = StarLayerGraph()
staff.parse(data="""@prefix ex: <http://example.org/> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Chair rdfs:subClassOf ex:Member . ex:bo a ex:Chair . ex:cy a ex:Clerk . ex:di a ex:Clerk .""", format="turtle")

def names(result):
    """The ?x values of a result in order, to keep the prints short."""
    return sorted(row.x.value for row in result)

# initNs: the prefix is bound once when the query is made, so the text needs no PREFIX line.
# entailment: the regime is bound once as well, so every run of this query uses RDFS.
membersQuery = StarQuery("SELECT ?x WHERE { ?x a ex:Member }", initNs={"ex": EX}, entailment=ENTAILMENT.RDFS)

# One query object runs against both graphs. Under RDFS a Chair is a Member, so bo is found.
result = membersQuery.run(people)
print("members of people:", names(result))
result = membersQuery.run(staff)
print("members of staff:", names(result))

# A single run can set entailment to None. bo is no longer found: nothing else says a Chair is a Member.
result = membersQuery.run(staff, entailment=None)
print("members of staff, entailment=None:", names(result))

# initBindings: variables that are bound before the query runs. The values are terms, not prefixed names.
# Given in run(), they apply to that run only, so one query text can be asked about each class in turn.
classQuery = StarQuery("SELECT ?x WHERE { ?x a ?class }")
result = classQuery.run(staff, initBindings={"class": NamedNode(EX + "Chair")})
print("chairs of staff:", names(result))
result = classQuery.run(staff, initBindings={"class": NamedNode(EX + "Clerk")})
print("clerks of staff:", names(result))

# A syntax error is raised when the query is made, not when it is first run.
try:
    StarQuery("SELECT ?x WHERE { ?x a }")
except ValueError as error:
    print("rejected when made:", str(error).splitlines()[0])


members of people: ['http://example.org/ann']
members of staff: ['http://example.org/bo']
members of staff, entailment=None: []
chairs of staff: ['http://example.org/bo']
clerks of staff: ['http://example.org/cy', 'http://example.org/di']
rejected when made: SPARQL syntax error at line 1, column 24: expected an RDF term or variable (found '}')


## 11. SPARQL 1.2 Update

`update()` runs SPARQL Update, including `INSERT DATA` and `DELETE DATA` with SPARQL 1.2 syntax and triple-term patterns. Like `query()`, it uses the graph's prefixes and takes `custom_functions` and `custom_aggregate_functions`.


In [15]:
g_update = StarLayerGraph(prefixes={"ex": EX, "rdf": RDF})

# INSERT DATA with a ground triple term - the base triple is not asserted.
g_update.update("""
    INSERT DATA { ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")
print("triples:", len(g_update))
print()
print(g_update.serialize(format="turtle"))

triples: 1

@version "1.2" .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



In [16]:
# INSERT ... WHERE: a triple-term pattern with variables, matched first,
# then instantiated into the INSERT template.
g_update.update("""
    INSERT { ?stmt ex:confidence "high" }
    WHERE  { ?stmt rdf:reifies <<( ?s ex:worksFor ?o )>> }
""")
print(g_update.serialize(format="turtle"))

@version "1.2" .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:claim ex:confidence "high" ;
	rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



### 11.1 Updates stay in their graph

An update on a `StarLayerGraph` stays within its graph: what it leaves unqualified is aimed at the graph, and an update that names another graph raises `ValueError`. An update on a `StarLayerDataset` acts on the whole store.

In [17]:
from starlayer.graph import StarLayerDataset

dataset = StarLayerDataset(prefixes={"ex": EX})
people = dataset.graph(EX + "people")  # a graph of the dataset uses its prefixes

# no GRAPH block: the triple goes in the people graph
people.update("INSERT DATA { ex:ann ex:knows ex:bo }")
# DELETE ... WHERE reads and deletes in the people graph
people.update("DELETE { ?s ex:knows ?o } INSERT { ?s ex:met ?o } WHERE { ?s ex:knows ?o }")
print(people.serialize(format="nt"))

# naming another graph is refused
try:
    people.update("INSERT DATA { GRAPH ex:other { ex:ann ex:knows ex:cy } }")
except ValueError as error:
    print("refused:", error)

# the dataset has no such limit
dataset.update("INSERT DATA { GRAPH ex:other { ex:ann ex:knows ex:cy } }")
print(len(dataset), "triples in the dataset;", len(people), "in the people graph")


<http://example.org/ann> <http://example.org/met> <http://example.org/bo> .

refused: this StarLayerGraph holds graph <http://example.org/people>; update cannot write to graph <http://example.org/other>
2 triples in the dataset; 1 in the people graph


## 12. Your own functions and aggregates

`query()` and `update()` take `custom_functions` and `custom_aggregate_functions`, Oxigraph's own hooks, for the one call. Each maps a function IRI (a `NamedNode` or its text) to Python. A function takes pyoxigraph terms and returns a term, or `None` to leave the variable unbound; the query then calls it as `<iri>(?x)`. An aggregate is a factory for an object with `accumulate(term)` and `finish()`. A query that names a function that was not passed raises a `RuntimeError` with Oxigraph's "custom function ... is not supported".

In [18]:
from starlayer.graph import Literal

def shout(term):
    return Literal(term.value.upper())

class Join:  # an aggregate: it sees each value once, then gives the result
    def __init__(self):
        self.parts = []
    def accumulate(self, term):
        self.parts.append(term.value)
    def finish(self):
        return Literal(", ".join(sorted(self.parts)))

g_fn = StarLayerGraph()
g_fn.parse(data='''@prefix ex: <http://example.org/> .
ex:teamA ex:name "ann", "al" .
ex:teamB ex:name "bo" .''', format="turtle")

rows = g_fn.query('SELECT ?team (<urn:fn:shout>(?n) AS ?loud) WHERE { ?team ex:name ?n }', custom_functions={"urn:fn:shout": shout})
print(sorted((r.team.value, r.loud.value) for r in rows))

rows = g_fn.query('SELECT ?team (<urn:fn:join>(?n) AS ?names) WHERE { ?team ex:name ?n } GROUP BY ?team',
                  custom_aggregate_functions={"urn:fn:join": Join})
print(sorted((r.team.value, r.names.value) for r in rows))

# a function that was not passed
try:
    list(g_fn.query('SELECT (<urn:fn:shout>(?n) AS ?loud) WHERE { ?team ex:name ?n }'))
except RuntimeError as error:
    print("not passed:", error)

[('http://example.org/teamA', 'AL'), ('http://example.org/teamA', 'ANN'), ('http://example.org/teamB', 'BO')]
[('http://example.org/teamA', 'al, ann'), ('http://example.org/teamB', 'bo')]
not passed: The custom function <urn:fn:shout> is not supported


## 13. Checking a query

`check_query(text)` checks a SPARQL 1.2 query or update before it is run, so errors or inconsistencies can be identified before the query or update is executed. It checks the grammar and the `VERSION` line (section 8), warns about likely mistakes, such as a variable that is never bound, and runs a query against an empty Oxigraph, which finds what Oxigraph itself rejects. It never runs the query on your data.

It returns a `QueryCheck`: `ok` is false when there is an error, and `problems` lists everything found, each with a kind, a severity (`"error"` or `"warning"`) and a message.

A query may use prefixes that are defined by the graph that will be used with the query. Without `prefixes=`, an undeclared prefix is returned as a warning. To check against a graph's prefixes, use the graph's own method, `graph.check_query(text)`, or pass them: `check_query(text, prefixes=graph.prefixes)`.

A query that calls your own functions needs them passed, as for `query()`: `check_query(text, custom_functions=...)`, because the empty Oxigraph does not know them otherwise (section 12).

In [19]:
from starlayer.sparql import check_query

def show(check):
    print(check.form, "| ok" if check.ok else "| not ok")
    for problem in check.problems:
        where = f" (line {problem.line}, column {problem.column})" if problem.line else ""
        print("  ", problem.kind, problem.severity + ":", problem.message + where)

# check_query(text): the grammar, the VERSION line, Oxigraph and likely mistakes
PREFIX = "PREFIX ex: <http://example.org/> "
show(check_query(PREFIX + "SELECT ?s WHERE { ?s ex:name ?n }"))
show(check_query(PREFIX + "SELECT ?s WHERE { ?s ex:name }"))
show(check_query('VERSION "1.1" ' + PREFIX + "SELECT ?s WHERE { ?s ex:p <<( ex:a ex:b ex:c )>> }"))
# ?z is never bound: legal SPARQL, so a warning and the text is still ok
show(check_query(PREFIX + "SELECT ?z WHERE { ?s ex:name ?n }"))

# graph.check_query(text): the prefixes of the graph the text will run against are declared
g_prefixed = StarLayerGraph(prefixes={"ex": EX})
text = "SELECT ?s WHERE { ?s ex:name ?n }"
show(check_query(text))
show(g_prefixed.check_query(text))

# custom_functions=...: the empty Oxigraph does not know a function unless it is passed
text = "SELECT ?s WHERE { ?s ?p ?o FILTER(<urn:fn:unknown>(?o)) }"
show(check_query(text))
show(check_query(text, custom_functions={"urn:fn:unknown": lambda term: term}))


SELECT | ok
None | not ok
   syntax error: expected an RDF term or variable (found '}') (line 1, column 63)
SELECT | not ok
   version error: SPARQL query declares VERSION '1.1' but uses a triple term, which '1.1' conformance excludes (SPARQL 1.2 Query sec 4.3)
SELECT | ok
   shape warning: Projected variable z is never bound anywhere in this query's own sqe:where pattern.
SELECT | ok
   prefix warning: undeclared prefix 'ex' (found 'ex:name') (line 1, column 22)
SELECT | ok
SELECT | not ok
   engine error: The custom function <urn:fn:unknown> is not supported
SELECT | ok
